# Imports

In [0]:
from pyspark.sql import functions as F
from datetime import timedelta
from pyspark.sql import Window as W
from pyspark.sql.types import DecimalType, IntegerType, TimestampType, DateType, DoubleType

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

def rename_cols(df, mapping: dict):
    for old, new in mapping.items():
        if old in df.columns:
            df = df.withColumnRenamed(old, new)
    return df

# ft_consumidores para silver

In [0]:
df_bz = spark.table("bronze.ft_consumidores")

mapping = {
    "customer_id": "id_consumidor",
    "customer_zip_code_prefix": "prefixo_cep",
    "customer_city": "cidade",
    "customer_state": "estado",
}
df = rename_cols(df_bz, mapping)

# Tipagem
df = (df
      .withColumn("prefixo_cep", F.col("prefixo_cep").cast(IntegerType()))
      .withColumn("cidade", F.upper(F.col("cidade")))
      .withColumn("estado", F.upper(F.col("estado")))
     )

# Deduplicação por id_consumidor mantendo o mais recente pela ingestion_timestamp
win = W.partitionBy("id_consumidor").orderBy(F.col("ingestion_timestamp").desc())
df = (df
      .withColumn("rn", F.row_number().over(win))
      .filter(F.col("rn") == 1)
      .drop("rn"))

# Salvar
df.write.mode("overwrite").saveAsTable("silver.ft_consumidores")
print(f"silver.ft_consumidores: {df.count()} rows")

display(df.limit(10))

silver.ft_consumidores: 99441 rows


id_consumidor,customer_unique_id,prefixo_cep,cidade,estado,ingestion_timestamp
00012a2ce6f8dcda20d059ce98491703,248ffe10d632bebe4f7267f1f44844c9,6273,OSASCO,SP,2025-11-12T19:36:20.669Z
000161a058600d5901f007fab4c27140,b0015e09bb4b6e47c52844fab5fb6638,35550,ITAPECERICA,MG,2025-11-12T19:36:20.669Z
0001fd6190edaaf884bcaf3d49edf079,94b11d37cd61cb2994a194d11f89682b,29830,NOVA VENECIA,ES,2025-11-12T19:36:20.669Z
0002414f95344307404f0ace7a26f1d5,4893ad4ea28b2c5b3ddf4e82e79db9e6,39664,MENDONCA,MG,2025-11-12T19:36:20.669Z
000379cdec625522490c315e70c7a9fb,0b83f73b19c2019e182fd552c048a22c,4841,SAO PAULO,SP,2025-11-12T19:36:20.669Z
0004164d20a9e969af783496f3408652,104bdb7e6a6cdceaa88c3ea5fa6b2b93,13272,VALINHOS,SP,2025-11-12T19:36:20.669Z
000419c5494106c306a97b5635748086,14843983d4a159080f6afe4b7f346e7c,24220,NITEROI,RJ,2025-11-12T19:36:20.669Z
00046a560d407e99b969756e0b10f282,0b5295fc9819d831f68eb0e9a3e13ab7,20540,RIO DE JANEIRO,RJ,2025-11-12T19:36:20.669Z
00050bf6e01e69d5c0fd612f1bcfb69c,e3cf594a99e810f58af53ed4820f25e5,98700,IJUI,RS,2025-11-12T19:36:20.669Z
000598caf2ef4117407665ac33275130,7e0516b486e92ed3f3afdd6d1276cfbd,35540,OLIVEIRA,MG,2025-11-12T19:36:20.669Z


# t_pedidos para silver

In [0]:
df_bz = spark.table("bronze.ft_pedidos")

mapping = {
    "order_id": "id_pedido",
    "customer_id": "id_consumidor",
    "order_status": "status",
    "order_purchase_timestamp": "pedido_compra_timestamp",
    "order_approved_at": "pedido_aprovado_timestamp",
    "order_delivered_carrier_date": "pedido_carregado_timestamp",
    "order_delivered_customer_date": "pedido_entregue_timestamp",
    "order_estimated_delivery_date": "pedido_estimativa_entrega_timestamp",
}
df = rename_cols(df_bz, mapping)

# Tipagem
ts_cols = ["pedido_compra_timestamp","pedido_aprovado_timestamp",
           "pedido_carregado_timestamp","pedido_entregue_timestamp",
           "pedido_estimativa_entrega_timestamp"]
for c in ts_cols:
    df = df.withColumn(c, F.col(c).cast(TimestampType()))

# Tradução de status
status_map = {
    "delivered": "entregue",
    "invoiced": "faturado",
    "shipped": "enviado",
    "processing": "em processamento",
    "unavailable": "indisponível",
    "canceled": "cancelado",
    "created": "criado",
    "approved": "aprovado",
}
mapping_expr = F.create_map([F.lit(x) for kv in status_map.items() for x in kv])
df = df.withColumn("status", F.coalesce(mapping_expr[F.col("status")], F.col("status")))

# Derivadas
df = df.withColumn(
    "tempo_entrega_dias",
    F.when(F.col("pedido_entregue_timestamp").isNotNull(),
           F.datediff(F.col("pedido_entregue_timestamp"), F.col("pedido_compra_timestamp")))
)
df = df.withColumn(
    "tempo_entrega_estimado_dias",
    F.when(F.col("pedido_estimativa_entrega_timestamp").isNotNull(),
           F.datediff(F.col("pedido_estimativa_entrega_timestamp"), F.col("pedido_compra_timestamp")))
)
df = df.withColumn(
    "diferenca_entrega_dias",
    F.when(F.col("tempo_entrega_dias").isNotNull() & F.col("tempo_entrega_estimado_dias").isNotNull(),
           F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias"))
)
df = df.withColumn(
    "entrega_no_prazo",
    F.when(F.col("pedido_entregue_timestamp").isNull(), F.lit("Não Entregue"))
     .when(F.col("diferenca_entrega_dias") <= 0, F.lit("Sim"))
     .otherwise(F.lit("Não"))
)

df.write.mode("overwrite").saveAsTable("silver.ft_pedidos")
print(f"silver.ft_pedidos: {df.count()} rows")

display(df.limit(10))

silver.ft_pedidos: 99441 rows


id_pedido,id_consumidor,status,pedido_compra_timestamp,pedido_aprovado_timestamp,pedido_carregado_timestamp,pedido_entregue_timestamp,pedido_estimativa_entrega_timestamp,ingestion_timestamp,tempo_entrega_dias,tempo_entrega_estimado_dias,diferenca_entrega_dias,entrega_no_prazo
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,entregue,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,2025-11-12T19:36:37.577Z,8,16,-8,Sim
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,entregue,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,2025-11-12T19:36:37.577Z,14,20,-6,Sim
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,entregue,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,2025-11-12T19:36:37.577Z,9,27,-18,Sim
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,entregue,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,2025-11-12T19:36:37.577Z,14,27,-13,Sim
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,entregue,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,2025-11-12T19:36:37.577Z,3,13,-10,Sim
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,entregue,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,2025-11-12T19:36:37.577Z,17,23,-6,Sim
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,faturado,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z,2025-11-12T19:36:37.577Z,null,28,null,Não Entregue
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,entregue,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,2025-11-12T19:36:37.577Z,10,22,-12,Sim
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,entregue,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z,2025-11-12T19:36:37.577Z,10,42,-32,Sim
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,entregue,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z,2025-11-12T19:36:37.577Z,18,25,-7,Sim


# ft_itens_pedidos para silver

In [0]:
df_bz = spark.table("bronze.ft_itens_pedidos")

mapping = {
    "order_id": "id_pedido",
    "order_item_id": "id_item",
    "product_id": "id_produto",
    "seller_id": "id_vendedor",
    "price": "preco_brl",
    "freight_value": "preco_frete",
}
df = rename_cols(df_bz, mapping)

df = (df
      .withColumn("id_item", F.col("id_item").cast(IntegerType()))
      .withColumn("preco_brl", F.col("preco_brl").cast(DecimalType(12,2)))
      .withColumn("preco_frete", F.col("preco_frete").cast(DecimalType(12,2)))
     )

df.write.mode("overwrite").saveAsTable("silver.ft_itens_pedidos")
print(f"silver.ft_itens_pedidos: {df.count()} rows")

display(df.limit(10))


silver.ft_itens_pedidos: 112650 rows


id_pedido,id_item,id_produto,id_vendedor,shipping_limit_date,preco_brl,preco_frete,ingestion_timestamp
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.90,13.29,2025-11-12T19:36:28.989Z
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.90,19.93,2025-11-12T19:36:28.989Z
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.00,17.87,2025-11-12T19:36:28.989Z
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79,2025-11-12T19:36:28.989Z
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.90,18.14,2025-11-12T19:36:28.989Z
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23T03:55:27.000Z,21.90,12.69,2025-11-12T19:36:28.989Z
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14T12:10:31.000Z,19.90,11.85,2025-11-12T19:36:28.989Z
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10T12:30:45.000Z,810.00,70.75,2025-11-12T19:36:28.989Z
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26T18:31:29.000Z,145.95,11.65,2025-11-12T19:36:28.989Z
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06T14:10:56.000Z,53.99,11.40,2025-11-12T19:36:28.989Z


# ft_pagamentos_pedidos para silver

In [0]:
df_bz = spark.table("bronze.ft_pagamentos_pedidos")

mapping = {
    "order_id": "id_pedido",
    "payment_sequential": "codigo_pagamento",
    "payment_type": "forma_pagamento",
    "payment_installments": "parcelas",
    "payment_value": "valor_pagamento",
}
df = rename_cols(df_bz, mapping)

pay_map = {
    "credit_card": "Cartão de Crédito",
    "boleto": "Boleto",
    "voucher": "Voucher",
    "debit_card": "Cartão de Débito",
}
mapping_expr = F.create_map([F.lit(x) for kv in pay_map.items() for x in kv])
df = df.withColumn("forma_pagamento", F.coalesce(mapping_expr[F.col("forma_pagamento")], F.lit("Outro")))

df = (df
      .withColumn("codigo_pagamento", F.col("codigo_pagamento").cast(IntegerType()))
      .withColumn("parcelas", F.col("parcelas").cast(IntegerType()))
      .withColumn("valor_pagamento", F.col("valor_pagamento").cast(DecimalType(12,2)))
     )

df.write.mode("overwrite").saveAsTable("silver.ft_pagamentos_pedidos")
print(f"silver.ft_pagamentos_pedidos: {df.count()} rows")

display(df.limit(10))

silver.ft_pagamentos_pedidos: 103886 rows


id_pedido,codigo_pagamento,forma_pagamento,parcelas,valor_pagamento,ingestion_timestamp
b81ef226f3fe1789b1e8b2acac839d17,1,Cartão de Crédito,8,99.33,2025-11-12T19:36:31.729Z
a9810da82917af2d9aefd1278f1dcfa0,1,Cartão de Crédito,1,24.39,2025-11-12T19:36:31.729Z
25e8ea4e93396b6fa0d3dd708e76c1bd,1,Cartão de Crédito,1,65.71,2025-11-12T19:36:31.729Z
ba78997921bbcdc1373bb41e913ab953,1,Cartão de Crédito,8,107.78,2025-11-12T19:36:31.729Z
42fdf880ba16b47b59251dd489d4441a,1,Cartão de Crédito,2,128.45,2025-11-12T19:36:31.729Z
298fcdf1f73eb413e4d26d01b25bc1cd,1,Cartão de Crédito,2,96.12,2025-11-12T19:36:31.729Z
771ee386b001f06208a7419e4fc1bbd7,1,Cartão de Crédito,1,81.16,2025-11-12T19:36:31.729Z
3d7239c394a212faae122962df514ac7,1,Cartão de Crédito,3,51.84,2025-11-12T19:36:31.729Z
1f78449c87a54faf9e96e88ba1491fa9,1,Cartão de Crédito,6,341.09,2025-11-12T19:36:31.729Z
0573b5e23cbd798006520e1d5b4c6714,1,Boleto,1,51.95,2025-11-12T19:36:31.729Z


# ft_avaliacoes_pedidos silver

“ID incorreto”:
Consideramos ID incorreto quando:

id_pedido é nulo ou id_pedido não existe em bronze.ft_pedidos (checado por anti-join contra order_id, com referência id_pedido_ref).
Essas linhas são descartadas ao final (apenas inner join com a referência é mantido).

“Data preenchida errada”:
Consideramos data preenchida errada quando ocorre qualquer uma das condições:

data_comentario é nula (inclui textos que não puderam ser parseados);

data_comentario está no futuro em relação à data atual (current_date());

data_resposta existe e sua data (to_date(data_resposta)) está no futuro.
Linhas que violam qualquer uma dessas regras são removidas.

In [0]:
df_bz = spark.table("bronze.ft_avaliacoes_pedidos")
pedidos_ref = (spark.table("bronze.ft_pedidos")
               .select("order_id").withColumnRenamed("order_id", "id_pedido_ref").distinct())

mapping = {
    "review_id": "id_avaliacao",
    "order_id": "id_pedido",
    "review_score": "avaliacao",
    "review_comment_title": "titulo_comentario",
    "review_comment_message": "comentario",
    "review_creation_date": "data_comentario",
    "review_answer_timestamp": "data_resposta",
}
df = rename_cols(df_bz, mapping)

df = (df
      .withColumn("data_comentario_str", F.col("data_comentario").cast("string"))
      .withColumn("data_resposta_str",  F.col("data_resposta").cast("string"))
     )

df = df.withColumn(
    "data_comentario_ts",
    F.coalesce(
        F.expr("try_to_timestamp(data_comentario_str, 'yyyy-MM-dd HH:mm:ss')"),
        F.expr("try_to_timestamp(data_comentario_str, 'yyyy-MM-dd')"),
        F.expr("try_to_timestamp(data_comentario_str, 'dd/MM/yyyy HH:mm:ss')"),
        F.expr("try_to_timestamp(data_comentario_str, 'dd/MM/yyyy')"),
        F.expr("try_cast(data_comentario_str as timestamp)")
    )
).withColumn(
    "data_comentario",
    F.to_date("data_comentario_ts").cast(DateType())
).drop("data_comentario_ts")

df = df.withColumn(
    "data_resposta",
    F.coalesce(
        F.expr("try_to_timestamp(data_resposta_str, 'yyyy-MM-dd HH:mm:ss')"),
        F.expr("try_to_timestamp(data_resposta_str, 'yyyy-MM-dd')"),
        F.expr("try_to_timestamp(data_resposta_str, 'dd/MM/yyyy HH:mm:ss')"),
        F.expr("try_to_timestamp(data_resposta_str, 'dd/MM/yyyy')"),
        F.expr("try_cast(data_resposta_str as timestamp)")
    ).cast(TimestampType())
).drop("data_resposta_str")

# Regras de qualidade
today = F.current_date()
cond_bad_date = (
    F.col("data_comentario").isNull() |
    (F.col("data_comentario") > today) |
    (F.col("data_resposta").isNotNull() & (F.to_date("data_resposta") > today))
)

# Contagens (para documentar no caderno)
cnt_total_in = df.count()
cnt_null_id  = df.filter(F.col("id_pedido").isNull()).count()
cnt_not_found = (
    df.filter(F.col("id_pedido").isNotNull())
      .join(pedidos_ref, df.id_pedido == pedidos_ref.id_pedido_ref, "left_anti")
      .count()
)
cnt_bad_date = df.filter(cond_bad_date).count()

print(f"[INPUT] total rows: {cnt_total_in}")
print(f"[RULE] invalid id_pedido (NULL): {cnt_null_id}")
print(f"[RULE] invalid id_pedido (not found in bronze.ft_pedidos): {cnt_not_found}")
print(f"[RULE] bad dates (null/future): {cnt_bad_date}")

# Remoção efetiva
df_valid = (
    df.join(pedidos_ref, df.id_pedido == pedidos_ref.id_pedido_ref, "inner")
      .filter(~cond_bad_date)
      .drop("id_pedido_ref")
)

cnt_out = df_valid.count()
print(f"[OUTPUT] kept rows: {cnt_out} | removed rows: {cnt_total_in - cnt_out}")

df_valid.write.mode("overwrite").saveAsTable("silver.ft_avaliacoes_pedidos")
display(df_valid.limit(10))

[INPUT] total rows: 104162
[RULE] invalid id_pedido (NULL): 2236
[RULE] invalid id_pedido (not found in bronze.ft_pedidos): 2702
[RULE] bad dates (null/future): 8832
[OUTPUT] kept rows: 95307 | removed rows: 8855


id_avaliacao,id_pedido,avaliacao,titulo_comentario,comentario,data_comentario,data_resposta,ingestion_timestamp,data_comentario_str
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18,2018-01-18T21:46:59.000Z,2025-11-12T19:36:34.418Z,2018-01-18 00:00:00
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10,2018-03-11T03:05:13.000Z,2025-11-12T19:36:34.418Z,2018-03-10 00:00:00
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17,2018-02-18T14:36:24.000Z,2025-11-12T19:36:34.418Z,2018-02-17 00:00:00
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21T22:02:06.000Z,2025-11-12T19:36:34.418Z,2017-04-21 00:00:00
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01,2018-03-02T10:26:53.000Z,2025-11-12T19:36:34.418Z,2018-03-01 00:00:00
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,null,null,2018-04-13,2018-04-16T00:39:37.000Z,2025-11-12T19:36:34.418Z,2018-04-13 00:00:00
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,null,null,2017-07-16,2017-07-18T19:30:34.000Z,2025-11-12T19:36:34.418Z,2017-07-16 00:00:00
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,null,null,2018-08-14,2018-08-14T21:36:06.000Z,2025-11-12T19:36:34.418Z,2018-08-14 00:00:00
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,null,null,2017-05-17,2017-05-18T12:05:37.000Z,2025-11-12T19:36:34.418Z,2017-05-17 00:00:00
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,2018-05-22,2018-05-23T16:45:47.000Z,2025-11-12T19:36:34.418Z,2018-05-22 00:00:00


# ft_produtos para silver

In [0]:
df_bz = spark.table("bronze.ft_produtos")

mapping = {
    "product_id": "id_produto",
    "product_category_name": "categoria_produto",
    "product_weight_g": "peso_produto_gramas",
    "product_length_cm": "comprimento_centimetros",
    "product_height_cm": "altura_centimetros",
    "product_width_cm": "largura_centimetros",
}
df = rename_cols(df_bz, mapping)

int_cols = ["peso_produto_gramas","comprimento_centimetros","altura_centimetros","largura_centimetros"]
for c in int_cols:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast(IntegerType()))

df.write.mode("overwrite").saveAsTable("silver.ft_produtos")
print(f"silver.ft_produtos: {df.count()} rows")

display(df.limit(10))

silver.ft_produtos: 32951 rows


id_produto,categoria_produto,product_name_lenght,product_description_lenght,product_photos_qty,peso_produto_gramas,comprimento_centimetros,altura_centimetros,largura_centimetros,ingestion_timestamp
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14,2025-11-12T19:36:40.366Z
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20,2025-11-12T19:36:40.366Z
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15,2025-11-12T19:36:40.366Z
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26,2025-11-12T19:36:40.366Z
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13,2025-11-12T19:36:40.366Z
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60,745,1,200,38,5,11,2025-11-12T19:36:40.366Z
732bd381ad09e530fe0a5f457d81becb,cool_stuff,56,1272,4,18350,70,24,44,2025-11-12T19:36:40.366Z
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56,184,2,900,40,8,40,2025-11-12T19:36:40.366Z
37cc742be07708b53a98702e77a21a02,eletrodomesticos,57,163,1,400,27,13,17,2025-11-12T19:36:40.366Z
8c92109888e8cdf9d66dc7e463025574,brinquedos,36,1156,1,600,17,10,12,2025-11-12T19:36:40.366Z


# ft_vendedores para silver

In [0]:
df_bz = spark.table("bronze.ft_vendedores")

mapping = {
    "seller_id": "id_vendedor",
    "seller_zip_code_prefix": "prefixo_cep",
    "seller_city": "cidade",
    "seller_state": "estado",
}
df = rename_cols(df_bz, mapping)

df = (df
      .withColumn("prefixo_cep", F.col("prefixo_cep").cast(IntegerType()))
      .withColumn("cidade", F.upper(F.col("cidade")))
      .withColumn("estado", F.upper(F.col("estado")))
     )

df.write.mode("overwrite").saveAsTable("silver.ft_vendedores")
print(f"silver.ft_vendedores: {df.count()} rows")

display(df.limit(10))

silver.ft_vendedores: 3095 rows


id_vendedor,prefixo_cep,cidade,estado,ingestion_timestamp
3442f8959a84dea7ee197c632cb2df15,13023,CAMPINAS,SP,2025-11-12T19:36:42.742Z
d1b65fc7debc3361ea86b5f14c68d2e2,13844,MOGI GUACU,SP,2025-11-12T19:36:42.742Z
ce3ad9de960102d0677a81f5d0bb7b2d,20031,RIO DE JANEIRO,RJ,2025-11-12T19:36:42.742Z
c0f3eea2e14555b6faeea3dd58c1b1c3,4195,SAO PAULO,SP,2025-11-12T19:36:42.742Z
51a04a8a6bdcb23deccc82b0b80742cf,12914,BRAGANCA PAULISTA,SP,2025-11-12T19:36:42.742Z
c240c4061717ac1806ae6ee72be3533b,20920,RIO DE JANEIRO,RJ,2025-11-12T19:36:42.742Z
e49c26c3edfa46d227d5121a6b6e4d37,55325,BREJAO,PE,2025-11-12T19:36:42.742Z
1b938a7ec6ac5061a66a3766e0e75f90,16304,PENAPOLIS,SP,2025-11-12T19:36:42.742Z
768a86e36ad6aae3d03ee3c6433d61df,1529,SAO PAULO,SP,2025-11-12T19:36:42.742Z
ccc4bbb5f32a6ab2b7066a4130f114e3,80310,CURITIBA,PR,2025-11-12T19:36:42.742Z


# dm_categoria_produtos_traducao para silver

In [0]:
df_bz = spark.table("bronze.dm_categoria_produtos_traducao")

mapping = {
    "product_category_name": "nome_produto_pt",
    "product_category_name_english": "nome_produto_en",
}
df = rename_cols(df_bz, mapping)

df.write.mode("overwrite").saveAsTable("silver.dm_categoria_produtos_traducao")
print(f"silver.dm_categoria_produtos_traducao: {df.count()} rows")

display(df.limit(10))

silver.dm_categoria_produtos_traducao: 71 rows


nome_produto_pt,nome_produto_en,ingestion_timestamp
beleza_saude,health_beauty,2025-11-12T19:36:44.938Z
informatica_acessorios,computers_accessories,2025-11-12T19:36:44.938Z
automotivo,auto,2025-11-12T19:36:44.938Z
cama_mesa_banho,bed_bath_table,2025-11-12T19:36:44.938Z
moveis_decoracao,furniture_decor,2025-11-12T19:36:44.938Z
esporte_lazer,sports_leisure,2025-11-12T19:36:44.938Z
perfumaria,perfumery,2025-11-12T19:36:44.938Z
utilidades_domesticas,housewares,2025-11-12T19:36:44.938Z
telefonia,telephony,2025-11-12T19:36:44.938Z
relogios_presentes,watches_gifts,2025-11-12T19:36:44.938Z


# dm_cotacao_dolar para silver

In [0]:
# Descobrir o intervalo de datas
minmax = (spark.table("bronze.dm_cotacao_dolar")
          .select(F.to_date(F.to_timestamp("dataHoraCotacao")).alias("d"))
          .agg(F.min("d").alias("min_d"), F.max("d").alias("max_d"))
          .first())
min_d, max_d = minmax.min_d, minmax.max_d

# Buffer para o primeiro fim de semana seja preenchido com a sexta anterior
start_with_buffer = min_d - timedelta(days=7)

# Construir um calendário diário (inclusivo)
date_span = (spark.range(1)
             .select(F.explode(F.sequence(F.lit(start_with_buffer), F.lit(max_d))).alias("data")))

# Normalizar as cotações da Bronze e escolher o fechamento
rates_raw = (spark.table("bronze.dm_cotacao_dolar")
             .withColumn("ts", F.to_timestamp("dataHoraCotacao"))
             .withColumn("data", F.to_date("ts"))
             .withColumn("cotacao_dolar", F.col("cotacaoCompra").cast(DoubleType()))
             .select("data", "ts", "cotacao_dolar")
             )

# Para cada data, pegar a última cotação do dia (fechamento)
w_close = W.partitionBy("data").orderBy(F.col("ts").desc())
rates_daily = (rates_raw
               .withColumn("rn", F.row_number().over(w_close))
               .filter(F.col("rn") == 1)
               .select("data", "cotacao_dolar"))

# Forward-fill sobre o calendário (fins de semana recebem a cotação da sexta)
w_ffill = W.partitionBy(F.lit(1)).orderBy("data").rowsBetween(W.unboundedPreceding, 0)
rates_full = (date_span.join(rates_daily, "data", "left")
              .withColumn("cotacao_dolar", F.last("cotacao_dolar", ignorenulls=True).over(w_ffill)))

# Manter apenas datas >= min_d e selecionar o schema final
df_silver = (rates_full
    .filter(F.col("data") >= F.lit(min_d))
    .select(
        F.round(F.col("cotacao_dolar"), 2).cast(DecimalType(12, 2)).alias("cotacao_dolar"),
        F.col("data").cast(DateType()).alias("data")
    )
)

# Salvar na Silver
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
df_silver.write.mode("overwrite").saveAsTable("silver.dm_cotacao_dolar")

# Checagens
print("Rows:", df_silver.count(),
      "| NULL cotacao_dolar:", df_silver.filter(F.col("cotacao_dolar").isNull()).count(),
      "| range:", df_silver.agg(F.min("data"), F.max("data")).first())

display(df_silver.orderBy("data").limit(10))

Rows: 1458 | NULL cotacao_dolar: 0 | range: Row(min(data)=datetime.date(2016, 1, 4), max(data)=datetime.date(2019, 12, 31))


cotacao_dolar,data
4.04,2016-01-04
4.01,2016-01-05
4.03,2016-01-06
4.05,2016-01-07
4.02,2016-01-08
4.02,2016-01-09
4.02,2016-01-10
4.01,2016-01-11
4.03,2016-01-12
3.99,2016-01-13


# Verificações de integridade referencial

In [0]:
pedidos = spark.table("silver.ft_pedidos")
consumidores = spark.table("silver.ft_consumidores")
itens = spark.table("silver.ft_itens_pedidos")

# Pedidos órfãos (sem consumidor)
pedidos_orfaos = pedidos.join(consumidores.select("id_consumidor").distinct(),
                              "id_consumidor", "left_anti")
n_pedidos_orfaos = pedidos_orfaos.count()
print(f"Pedidos órfãos: {n_pedidos_orfaos}")

pedidos_ok = pedidos.join(consumidores.select("id_consumidor").distinct(),
                          "id_consumidor", "inner")

# Itens órfãos (sem pedido)
itens_orfaos = itens.join(pedidos_ok.select("id_pedido").distinct(),
                          "id_pedido", "left_anti")
n_itens_orfaos = itens_orfaos.count()
print(f"Itens órfãos: {n_itens_orfaos}")

itens_ok = itens.join(pedidos_ok.select("id_pedido").distinct(), "id_pedido", "inner")

# Persistir versões sem órfãos
pedidos_ok.write.mode("overwrite").saveAsTable("silver.ft_pedidos")
itens_ok.write.mode("overwrite").saveAsTable("silver.ft_itens_pedidos")

Pedidos órfãos: 0
Itens órfãos: 0


# silver.ft_pedido_total

In [0]:
# Bases
orders = (spark.table("silver.ft_pedidos")
          .select("id_pedido", "id_consumidor", "status", "pedido_compra_timestamp")
          .withColumn("data_pedido", F.to_date("pedido_compra_timestamp")))

payments = spark.table("silver.ft_pagamentos_pedidos").select("id_pedido", "valor_pagamento")
usd = spark.table("silver.dm_cotacao_dolar").select(
    F.col("data").alias("data_pedido"),
    F.col("cotacao_dolar")
)

# Pagamentos agregados por pedido (BRL)
pay_agg = (payments
           .groupBy("id_pedido")
           .agg(F.sum("valor_pagamento").alias("valor_total_pago_brl")))

# Join pedidos + pagamentos + cotação do dia
ft = (orders
      .join(pay_agg, "id_pedido", "left")
      .join(usd, "data_pedido", "left"))

# Tipagem e arredondamento
ft = (ft
      .withColumn("valor_total_pago_brl",
                  F.round(F.col("valor_total_pago_brl"), 2).cast(DecimalType(12, 2)))
      .withColumn("valor_total_pago_usd",
                  F.when(F.col("valor_total_pago_brl").isNotNull() & F.col("cotacao_dolar").isNotNull(),
                         F.round(
                             (F.col("valor_total_pago_brl").cast(DoubleType()) /
                              F.col("cotacao_dolar").cast(DoubleType())), 2
                         )
                  ).cast(DecimalType(12, 2)))
     )

# Seleção final
ft_final = ft.select(
    "id_pedido",
    "id_consumidor",
    "status",
    "valor_total_pago_brl",
    "valor_total_pago_usd",
    "data_pedido"
)

ft_final.write.mode("overwrite").saveAsTable("silver.ft_pedido_total")

print(f"silver.ft_pedido_total created with {ft_final.count()} rows")
display(ft_final.orderBy("data_pedido").limit(10))

silver.ft_pedido_total created with 99441 rows


id_pedido,id_consumidor,status,valor_total_pago_brl,valor_total_pago_usd,data_pedido
2e7a8482f6fb09756ca50c10d7bfc047,08c5351a6aca1c1589a38f244edeee9d,enviado,136.23,42.05,2016-09-04
e5fa5a7210941f7d56d0208e4e071d35,683c54fc24d40ee9f8a6fc179fd9856c,cancelado,75.06,22.95,2016-09-05
809a282bbd5dbcabb6f2f724fca862ec,622e13439d6b5a0b486c435618b2679e,cancelado,40.95,12.41,2016-09-13
bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,entregue,null,null,2016-09-15
71303d7e93b399f5bcd537d124c0bcfa,b106b360fe2ef8849fbbd056f777b4d5,cancelado,109.34,33.64,2016-10-02
cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,entregue,40.95,12.68,2016-10-03
be5bc2f0da14d8071e2d45451ad119d9,7ec40b22510fdbea1b08921dd39e63d8,entregue,39.09,12.10,2016-10-03
ef1b29b591d31d57c0d7337460dd83c9,dc607dc98d6a11d5d04d9f2a70aa6c34,entregue,92.27,28.57,2016-10-03
ae8a60e4b03c5a4ba9ca0672c164b181,e6f959bf384d1d53b6d68826699bba12,entregue,154.57,47.85,2016-10-03
d207cc272675637bfed0062edffd0818,b8cf418e97ae795672d326288dfab7a7,entregue,133.46,41.32,2016-10-03
